**qaoa_test_max_cut**
The primary objective of this notebook is to use Qiskit to simulate the QAOA algorithm, evaluating how it performs in both ideal (noiseless) and noisy environments. The notebook focuses on solving combinatorial optimization problems—specifically the Max-Cut problem—by translating them into an Ising Hamiltonian.

In [8]:
import sys
import os

# Adds the parent directory (root) to the search path
sys.path.append(os.path.abspath(os.path.join('..', '..')))

In [9]:
#imports
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from solver.quantum_solver.qaoa_solver.qaoa_mixers import add_ising_problem_ham, add_ising_mixer_ham

ModuleNotFoundError: No module named 'solver'

**Problem Definition**


In [ ]:

n = 5
s1, s2, s3, s4, s5 = sp.symbols('s1 s2 s3 s4 s5')
# We create a mock class to match the problem parser's expected structure
class MockIsing:
    def __init__(self, expr):
        self.expr = expr
        self.variables = list(expr.free_symbols)
        
cost_expr = s1*s5 + s2*s5 + s3*s5 + s4*s5
ising_problem = MockIsing(cost_expr)

In [ ]:
def build_qaoa_circuit(ising_problem, n):
    """Builds the base QAOA circuit with unbound parameters."""
    qc = QuantumCircuit(n)
    
    #Uniform superposition
    for i in range(n):
        qc.h(i)
        
    #We apply problem and mixing hamiltonians (using your functions!)
    qc, gamma_params = add_ising_problem_ham(qc, ising_problem, n)
    qc, beta_params = add_ising_mixer_ham(qc, ising_problem, n)
    
    return qc, gamma_params[0], beta_params[0]

In [ ]:
def run_single_QAOA(angles, qc, shots=1024):
    """Binds parameters and runs the circuit."""
    # Bind the angles to our parameters
    bound_qc = qc.assign_parameters(angles)
    
    # We must measure to get counts
    bound_qc.measure_all()
    
    sim = AerSimulator()
    result = sim.run(bound_qc, shots=shots).result()
    return result.get_counts()

In [ ]:
def get_expectation_value(counts, shots=1024):
    """Computes the average cost from the measurement counts."""
    expected_value = 0
    
    for bitstring, count in counts.items():
        # Convert Qiskit's '0'->1 and '1'->-1
        # Qiskit reads right-to-left: bitstring[-1] is qubit 0 (s1), bitstring[-5] is qubit 4 (s5)
        spins = [1 if b == '0' else -1 for b in reversed(bitstring)]
        
        # Calculate the cost: s1*s5 + s2*s5 + s3*s5 + s4*s5
        cost = (spins[0]*spins[4]) + (spins[1]*spins[4]) + (spins[2]*spins[4]) + (spins[3]*spins[4])
        expected_value += cost * count
        
    return expected_value / shots

In [ ]:
def plot_heatmap(ising_problem, n):
    """Evaluates the expectation value over a grid of gamma and beta values."""
    grid_size = 50
    gamma_vals = np.linspace(0, np.pi, grid_size)
    beta_vals = np.linspace(0, np.pi, grid_size)
    heatmap = np.zeros((grid_size, grid_size))
    
    # Build base circuit once
    qc, p_gamma, p_beta = build_qaoa_circuit(ising_problem, n)
    
    print("Generating heatmap... this will take a moment!")
    for i, b in enumerate(beta_vals):
        for j, g in enumerate(gamma_vals):
            # Pass the dictionary of parameters as requested
            angles = {p_gamma: g, p_beta: b}
            counts = run_single_QAOA(angles, qc.copy(), shots=512)
            heatmap[i, j] = get_expectation_value(counts, shots=512)
            
    # Plotting
    plt.figure(figsize=(8, 5))
    plt.imshow(heatmap, origin='lower', extent=[0, np.pi, 0, np.pi], aspect='auto', cmap='hot')
    plt.colorbar(label='Expectation Value (Cost)')
    plt.xlabel('Gamma ($\gamma$)')
    plt.ylabel('Beta ($\\beta$)')
    plt.title('QAOA Expectation Value Landscape')
    plt.show()

plot_heatmap(ising_problem, n)